In [ ]:
import cv2
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
labels = pd.read_csv("/content/labels.csv")

print("Columns:", labels.columns)
labels.columns = labels.columns.str.strip()
print(labels.head())

Columns: Index(['id', 'count'], dtype='object')
   id  count
0   1     35
1   2     41
2   3     41
3   4     44
4   5     41


In [ ]:


# ---- Load labels
labels = pd.read_csv("/content/labels.csv")
labels.columns = labels.columns.str.strip()

print("Columns:", labels.columns)
print(labels.head())

# ---- Check images
print("Sample files:", os.listdir("/content")[:10])

features = []
targets = []
names = []

# ---- Extract features
for i, row in labels.iterrows():
    filename = f"seq_{int(row['id']):06d}.jpg"
    path = "/content/" + filename   # ✅ FIXED PATH

    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        print("Image not found:", path)
        continue

    # Features
    avg_brightness = np.mean(img)
    variance = np.var(img)
    edges = cv2.Canny(img, 100, 200)
    edge_density = np.mean(edges)
    laplacian_var = cv2.Laplacian(img, cv2.CV_64F).var()

    features.append([avg_brightness, variance, edge_density, laplacian_var])
    targets.append(row["count"])
    names.append(filename)   # ✅ FIXED

if len(features) == 0:
    raise SystemExit("No images loaded. Check your folder path.")

# ---- Train Test Split
X = np.array(features, dtype=float)
y = np.array(targets, dtype=float)

X_train, X_test, y_train, y_test, names_train, names_test = train_test_split(
    X, y, names, test_size=0.2, random_state=42
)

# ---- Train Model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Training done with RandomForest!")

# ---- Evaluation
pred_test = model.predict(X_test)

mae = mean_absolute_error(y_test, pred_test)
mse = mean_squared_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

print("MAE:", round(mae, 2))
print("MSE:", round(mse, 2))
print("RMSE:", round(rmse, 2))

# ---- Feature Importance
feature_names = ["avg_brightness", "variance", "edge_density", "laplacian_var"]
importances = model.feature_importances_
idx = np.argsort(importances)[::-1]

plt.figure()
plt.bar(range(len(importances)), importances[idx])
plt.xticks(range(len(importances)), [feature_names[i] for i in idx], rotation=30)
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

# ---- Actual vs Predicted
plt.figure()
plt.scatter(y_test, pred_test)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.show()

# ---- Error Distribution
errors = pred_test - y_test

plt.figure()
sns.histplot(errors, kde=True)
plt.axvline(0, linestyle="--")
plt.title("Error Distribution")
plt.show()

# ---- Heatmap
df_feat = pd.DataFrame(X, columns=feature_names)
df_feat["Crowd Count"] = y

plt.figure()
sns.heatmap(df_feat.corr(), annot=True, cmap="coolwarm")
plt.title("Feature Correlation")
plt.show()

# ---- Show predictions on images
rows_test = []

for n, actual, predicted in zip(names_test, y_test, pred_test):
    rows_test.append([n, int(actual), int(predicted)])

    path_color = "/content/" + n   # ✅ FIXED
    img_color = cv2.imread(path_color)

    if img_color is None:
        print("Missing image:", path_color)
        continue

    cv2.putText(img_color, f"Actual: {int(actual)}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(img_color, f"Pred: {int(predicted)}", (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2_imshow(img_color)

# ---- Save results
pd.DataFrame(rows_test, columns=["image_name", "actual", "predicted"]).to_csv(
    "predicted_output.csv", index=False
)

print("Saved predictions to predicted_output.csv")